# Phase 2 — Augmentation Pipeline

Reads `processed_faces/faces_manifest.csv` and produces 4 versions of every aligned face crop:

| Augmentation | Purpose |
|---|---|
| `original` | Copy of aligned face — control group |
| `jpeg_compression` | q=30 roundtrip — simulates social media re-compression |
| `gaussian_blur` | 5×5 kernel — simulates low-quality camera blur |
| `gaussian_noise` | σ=25 — simulates sensor noise / low-light |

**Output**: `dataset/{aug}/{split}/{class}/{stem}_{aug}.jpg` + `dataset_manifest.csv` di root project.

## 1) Imports & paths

In [ ]:
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROCESSED = ROOT / "processed_faces"
SRC_MANIFEST = PROCESSED / "faces_manifest.csv"
OUT_ROOT = ROOT / "dataset"
OUT_MANIFEST = ROOT / "dataset_manifest.csv"

AUGMENTATIONS = ("original", "jpeg_compression", "gaussian_blur", "gaussian_noise")

np.random.seed(42)
print("ROOT:", ROOT)
print("Source manifest:", SRC_MANIFEST)

## 2) Augmentation functions

Tiap fungsi nulis hasil ke `dst` sebagai JPG quality=95. `baseline` cuma `shutil.copy` dari source asli.

In [ ]:
def write_original(src: Path, dst: Path) -> None:
    """Control group — copy aligned face apa adanya."""
    shutil.copy(src, dst)


def write_jpeg_compression(img: np.ndarray, dst: Path) -> None:
    """Encode JPEG q=30 lalu decode lagi -> simpan q=95.

    Hasilnya gambar yang sudah punya artifact JPEG (blocking/ringing) tapi
    disimpan dengan kualitas tinggi supaya storage roundtrip tidak nambah
    distorsi tambahan.
    """
    _, encoded = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 30])
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    cv2.imwrite(str(dst), decoded, [cv2.IMWRITE_JPEG_QUALITY, 95])


def write_gaussian_blur(img: np.ndarray, dst: Path) -> None:
    """Blur 5x5 — cukup untuk soften boundary artifact deepfake tanpa
    menghancurkan struktur wajah."""
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    cv2.imwrite(str(dst), blurred, [cv2.IMWRITE_JPEG_QUALITY, 95])


def write_gaussian_noise(img: np.ndarray, dst: Path) -> None:
    """Noise gaussian sigma=25 — kontras dengan blur (tambah info random
    vs hapus detail)."""
    noise = np.random.normal(0, 25, img.shape).astype(np.float32)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    cv2.imwrite(str(dst), noisy, [cv2.IMWRITE_JPEG_QUALITY, 95])

## 3) Siapkan output dirs

In [ ]:
df = pd.read_csv(SRC_MANIFEST)
print(f"Loaded manifest: {len(df)} rows")

for aug in AUGMENTATIONS:
    for split in ("train", "test"):
        for cls in ("real", "fake"):
            (OUT_ROOT / aug / split / cls).mkdir(parents=True, exist_ok=True)

print("Output dirs ready under:", OUT_ROOT)

## 4) Main loop — augment semua frame

In [ ]:
records: list[dict] = []
rows = df.to_dict("records")

for r in tqdm(rows, desc="augmenting"):
    rel = Path(r["path"])
    src = PROCESSED / rel
    stem = rel.stem
    split = r["split"]
    cls = r["class"]
    label = int(r["label"])

    img = cv2.imread(str(src))
    if img is None:
        print(f"WARNING: cannot read {src}")
        continue

    for aug in AUGMENTATIONS:
        dst = OUT_ROOT / aug / split / cls / f"{stem}_{aug}.jpg"
        if aug == "original":
            write_original(src, dst)
        elif aug == "jpeg_compression":
            write_jpeg_compression(img, dst)
        elif aug == "gaussian_blur":
            write_gaussian_blur(img, dst)
        elif aug == "gaussian_noise":
            write_gaussian_noise(img, dst)

        records.append(
            {
                "original_path": rel.as_posix(),
                "augmented_path": dst.relative_to(ROOT).as_posix(),
                "augmentation": aug,
                "label": label,
                "split": split,
                "class": cls,
            }
        )

out_df = pd.DataFrame(records)
out_df.to_csv(OUT_MANIFEST, index=False)
print(f"Wrote {len(out_df)} rows -> {OUT_MANIFEST}")

## 5) Sanity check — visualisasi sample

In [ ]:
import matplotlib.pyplot as plt

out_df = pd.read_csv(OUT_MANIFEST)
sample_stem = out_df["original_path"].iloc[0].split("/")[-1].replace(".jpg", "")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, aug in zip(axes, AUGMENTATIONS):
    rows_aug = out_df[(out_df["augmentation"] == aug) & out_df["augmented_path"].str.contains(sample_stem)]
    img_path = ROOT / rows_aug["augmented_path"].iloc[0]
    img = cv2.imread(str(img_path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(aug)
    ax.axis("off")
plt.tight_layout()
plt.show()